# Week 10 hands-on: the optimal fingerprint

This notebook reproduces every figure in the Week 10 chapter.  The world is a
synthetic zonal-mean temperature field: a polar-amplified forced trend of
2.5 K/century buried in internal variability made of a tropically-trapped
mode, a hemispherically asymmetric mode, a red large-scale background and a
little white noise.

The three lines that matter are in section 2:

```python
lam, E = np.linalg.eigh(Sigma_N)
q_opt  = E[:, :k] @ ((E[:, :k].T @ s_hat) / lam[:k])
d      = q_opt @ obs
```

Everything else is scaffolding.

## 0. Grid and the building blocks of the synthetic climate

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import legendre

rng = np.random.default_rng(20251116)

## 0. Grid and the building blocks of the synthetic climate

In [ ]:
nlat = 73
lat = np.linspace(-90, 90, nlat)
xl = np.sin(np.deg2rad(lat))                 # Legendre argument
wgt = np.cos(np.deg2rad(lat))                # area weight
wgt = wgt / wgt.sum()

ntime = 100                                  # years
nens = 30                                    # ensemble members
time = np.arange(ntime)

# --- forced response: polar-amplified warming, 2.5 K per century ---------
s_pat = 0.4 + 1.6 * xl**2                    # shape only
trend = 2.5 * time / ntime                   # global amplitude in time
forced = s_pat[:, None] * trend[None, :]     # (nlat, ntime), the truth

# --- internal variability: patterns and amplitudes ----------------------
p_enso = np.exp(-(lat / 12.0) ** 2)                      # tropically trapped
p_enso /= np.sqrt((p_enso**2).mean())
p_ann = np.tanh(lat / 20.0)                              # hemispheric see-saw
p_ann /= np.sqrt((p_ann**2).mean())

noise_pats = [p_enso, p_ann]
noise_amps = [1.50, 0.80]

for n in range(2, 40):                                   # red background
    pn = legendre(n)(xl)
    noise_pats.append(pn / np.sqrt((pn**2).mean()))
    noise_amps.append(0.45 * 6.0 / (n + 1))

noise_pats = np.array(noise_pats)                        # (nmode, nlat)
noise_amps = np.array(noise_amps)
sig_white = 0.15

# True internal-variability covariance (known analytically here)
Sigma_true = (noise_pats.T * noise_amps**2) @ noise_pats
Sigma_true += sig_white**2 * np.eye(nlat)


def draw_noise(n_real, nt, generator):
    """Return (n_real, nlat, nt) internal variability."""
    a = generator.normal(0, 1, size=(n_real, len(noise_amps), nt))
    a = a * noise_amps[None, :, None]
    out = np.einsum('rmt,ml->rlt', a, noise_pats)
    out += generator.normal(0, sig_white, size=(n_real, nlat, nt))
    return out

## 1. The large ensemble: signal = ensemble mean, noise = spread

With one Earth we never see the split in {eq}`eq148`.  A large ensemble
gives it to us: every member shares the forced response and draws its own
internal variability, so the ensemble mean estimates $\mathbf{s}$ and the
departures estimate $\mathbf{u}$.

In [ ]:
X = forced[None, :, :] + draw_noise(nens, ntime, rng)    # (nens, nlat, ntime)
Xbar = X.mean(axis=0)                                    # forced estimate
Xanom = X - Xbar[None, :, :]                             # internal estimate

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))

yr = ntime - 1
ax[0].plot(lat, X[:, :, yr].T, color='0.7', lw=0.9)
ax[0].plot(lat, X[0, :, yr], color='0.7', lw=0.9, label='ensemble members')
ax[0].plot(lat, Xbar[:, yr], 'k-', lw=2.2, label=r'ensemble mean $\hat{s}$')
ax[0].plot(lat, forced[:, yr], 'r--', lw=2.0, label='true forced response')
ax[0].set_xlabel('Latitude')
ax[0].set_ylabel(r'$\Delta T$ (K)')
ax[0].set_title('(a) Year 100: one snapshot of the ensemble')
ax[0].set_xlim(-90, 90)
ax[0].set_xticks(np.arange(-90, 91, 30))
ax[0].legend(fontsize=8, loc='upper center')
ax[0].grid(alpha=0.3)

jlat = np.argmin(abs(lat - 65))
ax[1].plot(time, X[:, jlat, :].T, color='0.7', lw=0.8)
ax[1].plot(time, Xbar[jlat, :], 'k-', lw=2.2, label=r'ensemble mean')
ax[1].plot(time, forced[jlat, :], 'r--', lw=2.0, label='true forced response')
ax[1].set_xlabel('Year')
ax[1].set_ylabel(r'$\Delta T$ (K)')
ax[1].set_title(r'(b) Time series at 65$^\circ$N')
ax[1].legend(fontsize=8, loc='upper left')
ax[1].grid(alpha=0.3)

fig.tight_layout()

## 2. Forced pattern, noise EOFs, and the optimal fingerprint

Estimate the forced response by regressing the ensemble mean on time, and
the noise covariance from an *independent* control integration.  Then
$\mathbf{q}_\star \propto \Sigma_N^{-1}\mathbf{s}$, truncated at $k$ noise EOFs.

Try changing `ktrunc` and watch the filter get wigglier.

In [ ]:
# signal pattern: regress the ensemble mean on time (K per century)
tt = (time - time.mean())
s_hat = (Xbar * tt).sum(axis=1) / (tt**2).sum() * ntime

# noise covariance from an INDEPENDENT control integration.  This is the
# realistic situation: a few tens of effectively independent samples to
# estimate a covariance matrix of dimension nlat.
n_ctl = 50
Z = draw_noise(1, n_ctl, rng)[0].T                       # (n_ctl, nlat)
Z = Z - Z.mean(axis=0, keepdims=True)
Sigma_N = (Z.T @ Z) / (n_ctl - 1.0)

lam, E = np.linalg.eigh(Sigma_N)
lam, E = lam[::-1], E[:, ::-1]

ktrunc = 12
q_opt = E[:, :ktrunc] @ ((E[:, :ktrunc].T @ s_hat) / lam[:ktrunc])


def nrm(v):
    return v / np.max(np.abs(v))


fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))

ax[0].plot(lat, nrm(s_hat), 'r-', lw=2.2, label=r'forced response $\hat{s}$')
ax[0].plot(lat, nrm(E[:, 0]), 'b-', lw=1.8,
           label=r'leading noise EOF ($\lambda_1$=%.2f)' % lam[0])
ax[0].plot(lat, nrm(E[:, 1]), 'c-', lw=1.4,
           label=r'2nd noise EOF ($\lambda_2$=%.2f)' % lam[1])
ax[0].axhline(0, color='k', lw=0.6)
ax[0].set_xlabel('Latitude')
ax[0].set_ylabel('normalised amplitude')
ax[0].set_title('(a) The signal lives where the noise lives')
ax[0].set_xlim(-90, 90)
ax[0].set_xticks(np.arange(-90, 91, 30))
ax[0].legend(fontsize=8, loc='lower center')
ax[0].grid(alpha=0.3)

ax[1].plot(lat, nrm(s_hat), 'r--', lw=1.6, label=r'$\hat{s}$ (naive filter)')
ax[1].plot(lat, nrm(q_opt), 'k-', lw=2.4,
           label=r'$\mathbf{q}=\Sigma_N^{-1}\hat{s}$ (optimal)')
ax[1].axhline(0, color='k', lw=0.6)
ax[1].set_xlabel('Latitude')
ax[1].set_ylabel('normalised weight')
ax[1].set_title(r'(b) The optimal filter is not the signal pattern')
ax[1].set_xlim(-90, 90)
ax[1].set_xticks(np.arange(-90, 91, 30))
ax[1].legend(fontsize=8, loc='lower center')
ax[1].grid(alpha=0.3)

fig.tight_layout()

## 3. Detection time series

Apply three filters to the *same* simulated world: the area-weighted global
mean, the raw signal pattern, and the optimal fingerprint.  Each index is
divided by its own internal-variability standard deviation, so the three
curves are directly comparable in signal-to-noise units.

In [ ]:
obs = forced + draw_noise(1, ntime, rng)[0]              # one "observed" world

filters = {
    'global mean': wgt.copy(),
    r'signal pattern $\hat{s}$': s_hat.copy(),
    r'optimal $\Sigma_N^{-1}\hat{s}$': q_opt.copy(),
}
colors = {'global mean': 'tab:green',
          r'signal pattern $\hat{s}$': 'tab:red',
          r'optimal $\Sigma_N^{-1}\hat{s}$': 'k'}

fig, ax = plt.subplots(figsize=(7.2, 4.6))
for name, q in filters.items():
    sd = np.sqrt(q @ Sigma_true @ q)                     # noise std of index
    d = (q @ obs) / sd                                   # index in units of sd
    ax.plot(time, d, color=colors[name], lw=2.0, label=name)
    snr = (q @ (s_pat * 2.5)) / sd
    print(f'{name:28s}  SNR at yr100 = {snr:6.2f}')

ax.axhspan(-2, 2, color='0.85', zorder=0)
ax.text(97, -1.75, r'$\pm 2\sigma$ of internal variability', fontsize=8,
        color='0.35', ha='right')
ax.axhline(0, color='k', lw=0.6)
ax.set_xlabel('Year')
ax.set_ylabel(r'detection variable  $d(t)/\sigma_d$')
ax.set_title('Same world, three filters')
ax.legend(fontsize=9, loc='upper left')
ax.grid(alpha=0.3)
fig.tight_layout()

print('Mahalanobis SNR of truth  =', np.sqrt((s_pat * 2.5) @
                                             np.linalg.solve(Sigma_true,
                                                             s_pat * 2.5)))

## 4. Truncation: apparent vs actual signal-to-noise

The inverse in $\Sigma_N^{-1}$ divides by the smallest eigenvalues, which are
exactly the ones a short control run estimates worst.  Scoring the filter
with the same sample used to build it (red) sails straight through the
theoretical ceiling.  Verified against the truth (black), skill flattens
and then decays.

In [ ]:
s_true = s_pat * 2.5
ks = np.arange(1, n_ctl)
apparent, actual = [], []
for k in ks:
    proj = E[:, :k].T @ s_hat
    apparent.append(np.sum(proj**2 / lam[:k]))
    qk = E[:, :k] @ (proj / lam[:k])
    actual.append((qk @ s_true) ** 2 / (qk @ Sigma_true @ qk))

ceiling = s_true @ np.linalg.solve(Sigma_true, s_true)

fig, ax = plt.subplots(figsize=(7.2, 4.6))
ax.plot(ks, np.sqrt(apparent), 'r-o', ms=3, lw=1.8,
        label='apparent SNR (uses the same data twice)')
ax.plot(ks, np.sqrt(actual), 'k-o', ms=3, lw=1.8,
        label='actual SNR (verified against truth)')
ax.axhline(np.sqrt(ceiling), color='b', ls='--', lw=1.5,
           label=r'ceiling $\sqrt{s^{T}\Sigma_N^{-1}s}$')
ax.set_xlabel('number of retained noise EOFs, $k$')
ax.set_ylabel('signal-to-noise ratio')
ax.axvline(ktrunc, color='0.5', ls=':', lw=1.5)
ax.text(ktrunc + 1, 0.5, r'$k=%d$ used in Fig. 2$-$3' % ktrunc, fontsize=8,
        color='0.35')
ax.set_yscale('log')
ax.set_title(r'Truncation trade-off ($\Sigma_N$ from %d control years, '
             r'%d dimensions)' % (n_ctl, nlat))
ax.legend(fontsize=9, loc='lower right')
ax.grid(alpha=0.3)
fig.tight_layout()

## 5. Two-signal attribution: separability of forced components

$\mathrm{Cov}(\hat\beta) = (X^T\Sigma_N^{-1}X)^{-1}$ says that separating two
forcings depends on how *distinguishable* their patterns are after
prewhitening, not on how large they are.  Panel (b) is what degeneracy
looks like: the sum is well constrained, the split is not.

**Exercise.** Sweep `a_pat_deg` continuously from the separable pattern to
the GHG pattern and plot $\mathrm{sd}(\hat\beta_{GHG})$ against $\cos_M$.
Compare with the variance inflation factor $1/(1-\cos_M^2)$.

In [ ]:
g_pat = 0.4 + 1.6 * xl**2                                # "GHG": polar amplified
a_pat_sep = np.exp(-((lat - 45) / 22.0) ** 2)            # "aerosol": NH midlat
a_pat_sep /= np.sqrt((a_pat_sep**2).mean()) / 1.2
a_pat_deg = 0.5 + 1.5 * xl**2                            # near-degenerate twin
a_pat_deg /= np.sqrt((a_pat_deg**2).mean()) / 1.2

beta_true = np.array([1.0, -0.8])
Sinv = np.linalg.inv(Sigma_true)


def fit(Xmat, y, W):
    A = Xmat.T @ W @ Xmat
    return np.linalg.solve(A, Xmat.T @ W @ y)


def cosine_M(u, v):
    return (u @ Sinv @ v) / np.sqrt((u @ Sinv @ u) * (v @ Sinv @ v))


nreal = 400
res = {'sep_gls': [], 'sep_ols': [], 'deg_gls': []}
for _ in range(nreal):
    e = draw_noise(1, 1, rng)[0][:, 0]
    Xs = np.column_stack([g_pat, a_pat_sep])
    Xd = np.column_stack([g_pat, a_pat_deg])
    res['sep_gls'].append(fit(Xs, Xs @ beta_true + e, Sinv))
    res['sep_ols'].append(fit(Xs, Xs @ beta_true + e, np.eye(nlat)))
    res['deg_gls'].append(fit(Xd, Xd @ beta_true + e, Sinv))
res = {k: np.array(v) for k, v in res.items()}

fig, ax = plt.subplots(1, 2, figsize=(11, 4.6))

ax[0].scatter(res['sep_ols'][:, 0], res['sep_ols'][:, 1], s=9,
              color='tab:red', alpha=0.45, label='ordinary least squares')
ax[0].scatter(res['sep_gls'][:, 0], res['sep_gls'][:, 1], s=9,
              color='k', alpha=0.6, label=r'optimal (weighted by $\Sigma_N^{-1}$)')
ax[0].plot(beta_true[0], beta_true[1], 'b*', ms=16, label='truth')
ax[0].set_title(r'(a) distinct patterns, $\cos_M=%.2f$' %
                cosine_M(g_pat, a_pat_sep))
ax[0].set_xlim(-1, 3)
ax[0].set_ylim(-3, 1.5)

ax[1].scatter(res['deg_gls'][:, 0], res['deg_gls'][:, 1], s=9,
              color='k', alpha=0.6, label=r'optimal (weighted by $\Sigma_N^{-1}$)')
ax[1].plot(beta_true[0], beta_true[1], 'b*', ms=16, label='truth')
ax[1].set_title(r'(b) near-degenerate patterns, $\cos_M=%.3f$' %
                cosine_M(g_pat, a_pat_deg))
ax[1].set_xlim(-25, 27)
ax[1].set_ylim(-27, 25)

for a in ax:
    a.axhline(0, color='0.5', lw=0.8)
    a.axvline(0, color='0.5', lw=0.8)
    a.set_xlabel(r'$\beta_{\rm GHG}$')
    a.set_ylabel(r'$\beta_{\rm AER}$')
    a.legend(fontsize=8, loc='upper left')
    a.grid(alpha=0.3)

fig.tight_layout()

print('cos_M separable      =', cosine_M(g_pat, a_pat_sep))
print('cos_M near-degenerate=', cosine_M(g_pat, a_pat_deg))
print('sd beta_GHG sep gls  =', res['sep_gls'][:, 0].std())
print('sd beta_GHG sep ols  =', res['sep_ols'][:, 0].std())
print('sd beta_GHG deg gls  =', res['deg_gls'][:, 0].std())
print('done')